# Transformación de propiedades usadas


<a id="indice"></a>

## Índice

- [1. Carga y partición cronológica fija 80/20](#seccion-carga-particion-0)
- [2. Feature Selection inicial](#seccion-seleccion-inicial-1)
- [3. Corrección e imputación antes de Tukey](#seccion-preparacion-inicial-2)
- [4. Tratamiento de valores atípicos de la variable objetivo](#seccion-objetivo-outliers-3)
  - [4.1. Tukey sobre el Train ya preparado](#seccion-objetivo-outliers-4)
- [5. Variables predictoras](#seccion-features-transform-5)
  - [5.1. Feature Transformation — codificación](#seccion-features-transform-6)
    - [5.1.1. Codificación de district — One-Hot Encoding](#seccion-district-codificacion-7)
  - [5.2. Feature Extraction](#seccion-feature-extraction-8)
  - [5.3. Feature Selection final — constantes y duplicadas](#seccion-filtro-final-9)
  - [5.4. Feature Transformation — escala numérica](#seccion-analisis-escala-10)
    - [5.4.1. Análisis de escala y distribución de variables numéricas](#seccion-analisis-escala-11)
    - [5.4.2. Aplicación de la escala seleccionada](#seccion-decision-escala-12)
- [6. Artefactos de auditoría y trazabilidad](#seccion-persistencia-13)
- [7. Aplicación acumulativa a Test](#seccion-test-final-14)
- [8. Exportación de datasets transformados y persistencia de transformadores](#seccion-exportacion-final-15)
- [9. Resumen final de transformaciones y características del dataset](#seccion-resumen-final-16)

Actualizar después de agregar, eliminar, renombrar o mover secciones: `python Ventas/Propiedades_Usadas/scripts/actualizar_indice_transformacion.py` desde la raíz del proyecto. `--check` verifica la sincronización sin escribir.


<a id="seccion-carga-particion-0"></a>

## 1. Carga y partición cronológica fija 80/20


In [1]:
from pathlib import Path
import importlib
import sys
import warnings
from importlib.metadata import version

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

# Configuración única del paquete para ejecutar desde la raíz o el notebook.
carpeta_paquete = next((c for p in (Path.cwd(), *Path.cwd().parents)
    for c in (p, p / "Ventas/Propiedades_Usadas")
    if (c / "scripts/__init__.py").is_file()), None)
if carpeta_paquete is None:
    raise RuntimeError("Ejecutar el notebook dentro del proyecto Moscow_EDA.")
if str(carpeta_paquete) not in sys.path:
    sys.path.insert(0, str(carpeta_paquete))
from scripts.contrato_datasets import (
    CARPETA_PROYECTO, DATA_DIR as DATOS_PROYECTO, DATASETS, SCRIPTS_DIR, RAIZ as raiz,
)
carpeta_notebook = CARPETA_PROYECTO
from scripts import preparacion_train_test as pt
pt = importlib.reload(pt)  # Evitar configuraciones antiguas en un kernel reutilizado.
entrada = DATASETS["PropUsadas_Integrado.csv"]
from scripts.almacenamiento_mlflow import SesionTemporal, registrar_archivos, RUN_TRANSFORMACION
sesion = SesionTemporal()
with sesion.paso():
    artefactos = sesion.carpeta("transformacion")
    train_crudo, _test_reservado, resumen_particion, contrato_particion = pt.cargar_y_dividir(entrada, artefactos)
    trazabilidad_train = []
    contrato_transformacion = {"version_flujo": 3}
    display(resumen_particion)


               registros      desde      hasta  porcentaje  desviacion_pp
dataset_split                                                            
train              39998 2020-01-01 2025-01-16      79.996         -0.004
test               10002 2025-01-17 2026-04-29      20.004          0.004


<a id="seccion-seleccion-inicial-1"></a>

## 2. Feature Selection inicial

Se conservan únicamente las entradas definidas para el flujo. Las siguientes columnas
se excluyen de los predictores y permanecen solo en las copias crudas:

| Variable eliminada | Motivo de eliminación |
|---|---|
| `date_posted` | Se utiliza `date_posted_dt` como fecha de referencia. |
| `okrug` | Fuera de la lista de predictores definida para el flujo. |
| `lat` | Se priorizan otras medidas de ubicación y accesibilidad. |
| `lon` | Se priorizan otras medidas de ubicación y accesibilidad. |
| `metro_station` | aumenta considerablemente la complejidad del modelo OLS y el riesgo de sobreajuste. Además, su información puede estar parcialmente representada por variables numéricas como metro_distance_min y to_center_km. |
| `price_per_sqm` | Riesgo de fuga de información al derivarse del precio objetivo. |
| `year_month` | La información mensual se extrae de `date_posted_dt`. |
| `secondary_price_per_sqm` | Se predice el precio total, sin precios por unidad de superficie. |
| `newbuild_price_per_sqm` | Se predice el precio total, sin precios por unidad de superficie. |
| `newbuild_mom_change_pct` | Se conserva solo la variación mensual del mercado secundario. |
| `n_listings_secondary` | Fuera de la lista de predictores definida para el flujo. |
| `n_listings_newbuild` | Fuera de la lista de predictores definida para el flujo. |
| `cbr_key_rate_pct` | Redundancia financiera; se prioriza `mortgage_key_rate_at_listing`. |
| `avg_mortgage_rate_pct` | Redundancia financiera; se prioriza `mortgage_key_rate_at_listing`. |
| `metro_lat` | Se priorizan otras medidas de ubicación y accesibilidad. |
| `metro_lon` | Se priorizan otras medidas de ubicación y accesibilidad. |
| `metro_year_opened` | Año de apertura fuera del enfoque actual. |

Se conserva `mortgage_key_rate_at_listing` mediante su alias de entrada y `district`
para su codificación independiente.


In [2]:
with sesion.paso():
    train_actual = pt.seleccionar_variables(train_crudo)
    pt.registrar_etapa("seleccion", train_crudo, train_actual, trazabilidad_train)
    print("Predictoras seleccionadas:", pt.obtener_X(train_actual).shape[1])


Predictoras seleccionadas: 20


<a id="seccion-preparacion-inicial-2"></a>

## 3. Corrección e imputación

Sobre el Train seleccionado completo: corregir solo `total_area` si es menor que sala más
cocina; conservar las superficies individuales observadas; validar pisos y construcción.


Imputar numéricas con medianas, `secondary_mom_change_pct` con su media y binarias con su moda.
Las nominales usan `__FALTANTE__`. Todos esos parámetros se ajustan **antes de Tukey**.



In [3]:
with sesion.paso():
    anterior = train_actual
    preparador = pt.PreparacionInicial()
    train_actual = preparador.fit_transform(train_actual)
    pt.registrar_etapa("correccion_imputacion", anterior, train_actual, trazabilidad_train)
    resumen_calidad_train, auditoria_train = pt.auditar_preparacion(anterior, train_actual, preparador)
    train_preparado_antes_tukey = train_actual.copy(deep=True)
    print("Train preparado:", len(train_actual), "anuncios; totales corregidos:", resumen_calidad_train["superficies_corregidas"])
    display(pd.DataFrame(resumen_calidad_train["imputaciones_mom"]).T)


,imputados,faltantes_finales,media_train
secondary_mom_change_pct,678.0,0.0,0.733206


Train preparado: 39998 anuncios; totales corregidos: 186


<a id="seccion-objetivo-outliers-3"></a>

## 4. Tratamiento de valores atípicos de la variable objetivo

<a id="seccion-objetivo-outliers-4"></a>

### 4.1. Tukey sobre el Train ya preparado

Q1 y Q3 de `price_rub` usan interpolación lineal. Se eliminan filas completas con precios
fuera de Q1 − 1,5 × IQR y Q3 + 1,5 × IQR; las fronteras se conservan. El precio no se ha
imputado ni modificado.


In [4]:
with sesion.paso():
    anterior = train_actual
    train_actual, train_outliers_preparados, resumen_tukey = pt.filtrar_outliers_tukey_train(train_actual)
    pt.registrar_etapa("tukey", anterior, train_actual, trazabilidad_train, permite_filtrar=True)
    ids_retenidos = train_actual["id"].copy()
    auditoria_train["retenido_tras_tukey"] = auditoria_train["id"].isin(ids_retenidos)
    resumen_calidad_train["superficies_corregidas_retenidas"] = int(train_actual["surface_corrected"].sum())
    resumen_tukey_breve = {k: resumen_tukey[k] for k in ["q1", "q3", "iqr", "limite_inferior", "limite_superior",
                           "filas_iniciales", "outliers_eliminados", "porcentaje_eliminado", "filas_finales"]}
    display(pd.Series(resumen_tukey_breve, name="Tukey — solo Train").to_frame())


,Tukey — solo Train
q1,7.282000e+06
q3,1.594100e+07
iqr,8.659000e+06
limite_inferior,-5.706500e+06
limite_superior,2.892950e+07
filas_iniciales,3.999800e+04
outliers_eliminados,2.014000e+03
porcentaje_eliminado,5.035252e+00
filas_finales,3.798400e+04


<a id="seccion-features-transform-5"></a>

## 5. Variables predictoras

<a id="seccion-features-transform-6"></a>

### 5.1. Feature Transformation — codificación


In [5]:
with sesion.paso():
    anterior = train_actual
    codificador_base = pt.crear_codificador_base(preparador.columnas_numericas_)
    X_base = codificador_base.fit_transform(train_actual)


<a id="seccion-district-codificacion-7"></a>

#### 5.1.1. Codificación de district — One-Hot Encoding


In [6]:
with sesion.paso():
    codificador_district = pt.crear_codificador_district()
    X_district = codificador_district.fit_transform(train_actual[["district"]])
    train_actual = pt.reemplazar_X(train_actual, pt.integrar_caracteristicas(X_base, X_district))
    pt.registrar_etapa("codificacion", anterior, train_actual, trazabilidad_train)
    pt.validar_matriz(pt.obtener_X(train_actual), train_actual)
    print("district:", len(codificador_district.named_steps["onehot"].categories_[0]),
          "categorías;", X_district.shape[1], "columnas.")
    del X_base, X_district


district: 129 categorías; 128 columnas.


<a id="seccion-feature-extraction-8"></a>

### 5.2. Feature Extraction

Se agregan a la matriz codificada la tendencia mensual y el mes calendario One-Hot, a partir
de su fecha preservada. Se ajustan origen y categorías solo con los anuncios conservados.
Las marcas de dominio creadas en el apartado 3 ya están integradas; no se vuelven a calcular.


In [7]:
with sesion.paso():
    anterior = train_actual
    extractor_temporal = pt.ExtraccionTemporal()
    train_actual = extractor_temporal.fit_transform(train_actual)
    pt.registrar_etapa("extraccion", anterior, train_actual, trazabilidad_train)

    pt.validar_superficies(train_actual)


<a id="seccion-filtro-final-9"></a>

### 5.3. Feature Selection final — constantes y duplicadas

Estos filtros necesitan la matriz codificada y las derivadas; por esa dependencia técnica
se ajustan después de la extracción y antes del escalado. No incorporan variables descartadas
ni usan la respuesta. Se conservan el primer nombre y el orden de columnas no redundantes.


In [8]:
with sesion.paso():
    anterior = train_actual
    roles_candidatas = pt.roles_candidatas(preparador, codificador_base, codificador_district, extractor_temporal)
    filtro_final = pt.crear_filtro_final(roles_candidatas)
    train_actual = pt.reemplazar_X(train_actual, filtro_final.fit_transform(pt.obtener_X(train_actual)))
    pt.registrar_etapa("filtro_final", anterior, train_actual, trazabilidad_train)
    pt.validar_superficies(train_actual, exigir_columnas=False)
    pt.validar_matriz(pt.obtener_X(train_actual), train_actual)
    if len(train_actual) != resumen_tukey["filas_finales"]:
        raise ValueError("Se reincorporaron o perdieron registros después de Tukey.")


<a id="seccion-analisis-escala-10"></a>

### 5.4. Feature Transformation — escala numérica

<a id="seccion-analisis-escala-11"></a>

#### 5.4.1. Análisis de escala y distribución de variables numéricas

La tabla se construye con las columnas **presentes en `train_actual` después de selección,
preparación, Tukey, codificación y extracción**. Se excluyen auxiliares, objetivo, dummies,
binarias y marcas. La desviación descriptiva usa `ddof=1`. Los extremos IQR solo describen
la distribución; no generan un segundo filtro ni recortan predictoras.

Se registra la huella de la matriz anterior al escalado y sus ID. El análisis exige que
la etapa previa sea el filtro final; el ajuste comprueba la misma huella, filas y variables.
Reejecutar esta etapa sobre datos ya escalados detiene el flujo.

El contenedor conserva las columnas actuales y su orden; el `StandardScaler` interno recibe
únicamente las variables seleccionadas, no las dummies ni las binarias.


In [9]:
with sesion.paso():
    if not trazabilidad_train or trazabilidad_train[-1]["etapa"] != "filtro_final":
        raise ValueError("El análisis requiere el Train acumulado sin escalar, después del filtro final.")
    X_train = pt.obtener_X(train_actual)
    pt.validar_matriz(X_train, train_actual)
    columnas_modelo = X_train.columns.tolist()
    columnas_evaluadas = pt.columnas_para_escala(X_train, filtro_final.named_steps["nombres"].roles_salida_)
    origen_analisis_escala = {
        "dataframe": "pt.obtener_X(train_actual)", "etapa_previa": "filtro_final",
        "filas": len(X_train), "predictoras": X_train.shape[1],
        "columnas_evaluadas": columnas_evaluadas, "huella_matriz": joblib.hash(X_train),
        "ids_sha256": trazabilidad_train[-1]["ids_sha256"],
    }
    contrato_transformacion["origen_analisis_escala"] = origen_analisis_escala
    numericas_train = X_train[columnas_evaluadas]
    tabla_estadisticas = pd.DataFrame({
        "minimo": numericas_train.min(), "maximo": numericas_train.max(),
        "rango": numericas_train.max() - numericas_train.min(),
        "media": numericas_train.mean(), "mediana": numericas_train.median(),
        "desv_estandar": numericas_train.std(ddof=1),
        "Q1": numericas_train.quantile(.25), "Q3": numericas_train.quantile(.75),
        "asimetria": numericas_train.skew(),
    })
    iqr_numericas = tabla_estadisticas["Q3"] - tabla_estadisticas["Q1"]
    extremos = numericas_train.lt(tabla_estadisticas["Q1"] - 1.5 * iqr_numericas)
    extremos |= numericas_train.gt(tabla_estadisticas["Q3"] + 1.5 * iqr_numericas)
    tabla_estadisticas["extremos_IQR_pct"] = 100 * extremos.mean()
    with pd.option_context("display.max_columns", None, "display.width", 190):
        display(tabla_estadisticas.round(4))

    pt.registrar_etapa("analisis_escala", train_actual, train_actual, trazabilidad_train)


,minimo,maximo,rango,media,mediana,desv_estandar,Q1,Q3,asimetria,extremos_IQR_pct
total_area,24.40,131.80,107.40,66.2977,64.4000,16.9176,53.90,76.60,0.5910,1.3585
living_area,12.60,89.20,76.60,41.3984,40.1000,10.9697,33.40,47.90,0.6404,1.5454
kitchen_area,6.00,24.80,18.80,11.1815,11.1000,2.8260,9.20,13.10,0.2732,0.5213
rooms,1.00,5.00,4.00,2.0915,2.0000,0.9993,1.00,3.00,0.8461,0.0000
floor,1.00,25.00,24.00,7.4488,6.0000,5.5255,3.00,10.00,1.0169,3.4014
total_floors,5.00,25.00,20.00,13.9957,12.0000,6.5430,9.00,22.00,0.2726,0.0000
building_year,1935.00,2024.00,89.00,1987.1929,1988.0000,22.5807,1968.00,2008.00,-0.1860,0.0000
ceiling_height,2.50,3.50,1.00,2.7853,2.7000,0.2187,2.70,3.00,0.9334,2.1799
metro_distance_min,2.00,328.00,326.00,59.1332,41.0000,54.9999,24.00,73.00,1.9885,8.0508
to_center_km,0.07,70.40,70.33,16.9280,13.5800,12.7500,9.60,18.55,2.1205,9.6067


<a id="seccion-decision-escala-12"></a>

#### 5.4.2. Aplicación de la escala seleccionada

La reevaluación del Train actual mantiene el escalado solo del año de construcción. Se estandariza solo el año de
construcción para centrar su gran componente respecto del intercepto; se comprueba su efecto
sobre el condicionamiento. Las otras unidades conservan interpretación. No se selecciona
Min-Max: los métodos OLS sin penalización no exigen escalas iguales y los extremos comprimirían
la zona central. Ninguna escala lineal vuelve normal una distribución asimétrica.

La evidencia de cada fila se toma de la tabla recién calculada, no de salidas guardadas.
Para modelos regularizados habría que reevaluar el escalado dentro de los folds de Train.


In [10]:
with sesion.paso():
    if trazabilidad_train[-1]["etapa"] != "analisis_escala":
        raise ValueError("El escalado debe seguir al análisis vigente, sin reaplicaciones.")
    pt.verificar_estado_analizado(pt.obtener_X(train_actual), origen_analisis_escala["huella_matriz"], columnas_evaluadas, filtro_final.named_steps["nombres"].roles_salida_)
    pt.verificar_estado_analizado(X_train, origen_analisis_escala["huella_matriz"], columnas_evaluadas, filtro_final.named_steps["nombres"].roles_salida_)
    columnas_estandarizar = ["building_year"] if "building_year" in columnas_evaluadas else []
    columnas_normalizar = []
    columnas_escala_original = [c for c in columnas_evaluadas
                               if c not in columnas_estandarizar + columnas_normalizar]
    razones_original = {
        "total_area": "Coeficiente por m²; magnitud moderada y asimetría limitada.",
        "living_area": "Coeficiente por m² de sala; conservar interpretación.",
        "kitchen_area": "Coeficiente por m² de cocina; rango acotado.",
        "rooms": "Conteo discreto; interpretar por habitación.",
        "floor": "Conteo acotado; interpretar por piso, sin comprimir extremos.",
        "total_floors": "Conteo acotado; mantener unidad de piso.",
        "ceiling_height": "Conservar coeficiente por metro; OLS no exige Min-Max.",
        "metro_distance_min": "Cola derecha; Min-Max comprimiría el centro. Mantener minutos.",
        "to_center_km": "Asimetría elevada; escalar no la corrige. Mantener km.",
        "metro_to_center_km": "Rango moderado; mantener km para interpretar.",
        "mortgage_key_rate_at_listing": "Mantener interpretación por punto porcentual.",
        "months_since_start": "Tendencia ya referida al inicio; coeficiente por mes.",
        "secondary_mom_change_pct": "Unidad porcentual y magnitud moderada; no comprimir extremos.",
    }
    tabla_decisiones = pd.DataFrame([
        {"variable": c, "transformacion": "StandardScaler" if c in columnas_estandarizar else "Original",
         "justificacion": "Centrar el año respecto del intercepto y mejorar condición numérica."
                           if c in columnas_estandarizar else razones_original[c]}
        for c in columnas_evaluadas
    ])
    if (set(columnas_estandarizar) & set(columnas_normalizar)
            or set(columnas_estandarizar + columnas_normalizar + columnas_escala_original) != set(columnas_evaluadas)):
        raise ValueError("Los grupos de escalado no coinciden con las variables analizadas.")
    if tabla_decisiones["variable"].tolist() != columnas_evaluadas:
        raise ValueError("La tabla de decisiones pertenece a otro esquema.")
    for indice, fila in tabla_decisiones.iterrows():
        datos_variable = tabla_estadisticas.loc[fila["variable"]]
        tabla_decisiones.loc[indice, "justificacion"] += (
            f" Rango={datos_variable['rango']:.2f}; asimetría={datos_variable['asimetria']:.2f}.")
    print(tabla_decisiones.to_string(index=False))

    escalador = pt.EscaladoSelectivo(columnas_estandarizar, columnas_normalizar)
    X_train_sin_escalar = X_train
    X_train = escalador.fit_transform(X_train_sin_escalar)
    if escalador.huella_entrada_train_ != origen_analisis_escala["huella_matriz"]:
        raise ValueError("El escalador no se ajustó con la matriz analizada.")
    if escalador.filas_ajuste_ != origen_analisis_escala["filas"]:
        raise ValueError("El escalador utilizó otras observaciones.")
    if escalador.estandarizador_ is not None:
        if escalador.estandarizador_.feature_names_in_.tolist() != columnas_estandarizar:
            raise ValueError("StandardScaler recibió variables distintas de las seleccionadas.")
        np.testing.assert_allclose(escalador.estandarizador_.mean_, tabla_estadisticas.loc[columnas_estandarizar, "media"], rtol=1e-12)
        np.testing.assert_allclose(escalador.estandarizador_.var_, X_train_sin_escalar[columnas_estandarizar].var(ddof=0), rtol=1e-12)
    pt.validar_matriz(X_train, train_actual, columnas=columnas_modelo)
    np.testing.assert_allclose(X_train[columnas_estandarizar].mean(), 0, atol=1e-10)
    np.testing.assert_allclose(X_train[columnas_estandarizar].std(ddof=0), 1, atol=1e-10)
    if columnas_normalizar:
        assert X_train[columnas_normalizar].ge(-1e-12).all().all()
        assert X_train[columnas_normalizar].le(1 + 1e-12).all().all()
    columnas_conservadas = [c for c in X_train.columns if c not in columnas_estandarizar + columnas_normalizar]
    pd.testing.assert_frame_equal(X_train[columnas_conservadas], X_train_sin_escalar[columnas_conservadas])
    # Dos cifras de validación, sin agregar un análisis de modelos.
    diseno_antes = np.column_stack([np.ones(len(X_train)), X_train_sin_escalar.to_numpy(dtype=float)])
    diseno_despues = np.column_stack([np.ones(len(X_train)), X_train.to_numpy(dtype=float)])
    condicion_antes = float(np.linalg.cond(diseno_antes))
    condicion_despues = float(np.linalg.cond(diseno_despues))
    contrato_transformacion["escalado"] = {
        **pt.registro_escalado(escalador), "numericas_escala_original": columnas_escala_original,
        "condicion_train_con_intercepto_antes": condicion_antes,
        "condicion_train_con_intercepto_despues": condicion_despues,
        "desviacion_tabla_descriptiva_ddof": 1,
    }
    print("Grupos: estandarizadas", len(columnas_estandarizar), "; Min-Max", len(columnas_normalizar),
          "; numéricas en escala original", len(columnas_escala_original))
    print(f"Condición Train con intercepto: {condicion_antes:,.2f} → {condicion_despues:,.2f}")
    del X_train_sin_escalar, diseno_antes, diseno_despues

    anterior = train_actual
    train_actual = pt.reemplazar_X(train_actual, X_train)
    pt.registrar_etapa("escalado", anterior, train_actual, trazabilidad_train)
    pt.validar_superficies(train_actual, exigir_columnas=False)


                    variable transformacion                                                                                      justificacion
                  total_area       Original          Coeficiente por m²; magnitud moderada y asimetría limitada. Rango=107.40; asimetría=0.59.
                 living_area       Original                 Coeficiente por m² de sala; conservar interpretación. Rango=76.60; asimetría=0.64.
                kitchen_area       Original                          Coeficiente por m² de cocina; rango acotado. Rango=18.80; asimetría=0.27.
                       rooms       Original                           Conteo discreto; interpretar por habitación. Rango=4.00; asimetría=0.85.
                       floor       Original         Conteo acotado; interpretar por piso, sin comprimir extremos. Rango=24.00; asimetría=1.02.
                total_floors       Original                              Conteo acotado; mantener unidad de piso. Rango=20.00; asimetría=0.27.

<a id="seccion-persistencia-13"></a>

## 6. Artefactos de auditoría y trazabilidad

Los archivos de este apartado son de auditoría, no la entrega final para modelos.
La entrega oficial se genera en el apartado 8, después de transformar Test.
Se exporta `train_actual` y se persisten los objetos por etapa, reunidos también en
`transformadores_train.joblib`. `contrato_transformacion.json` v3 define el orden y el esquema.
Los CSV de trazabilidad verifican filas, identidad, respuesta y columnas en cada transición.

Los crudos conservan la fuente completa solo para auditoría. Los archivos preparados antes y
después de Tukey permiten comprobar la continuidad real. En validación temporal interna hay
que ajustar todo el flujo con el subtrain, aplicar Tukey solo allí y conservar la validación
completa. Cargar los CSV con `float_precision="round_trip"`. Los objetos anteriores requieren
el contrato v3; no deben combinarse con transformadores de ejecuciones previas.


In [11]:
with sesion.paso():
    pt.validar_matriz(pt.obtener_X(train_actual), train_actual)
    pd.testing.assert_frame_equal(train_actual[[*pt.AUXILIARES, pt.OBJETIVO]],
                                  train_crudo.loc[train_actual.index, [*pt.AUXILIARES, pt.OBJETIVO]])
    objetos_transformacion = {
        "preparacion": preparador, "base": codificador_base, "district": codificador_district,
        "extraccion": extractor_temporal, "filtro": filtro_final, "escalado": escalador,
        "columnas_X": columnas_modelo,
    }
    archivos_objetos = {"preparacion": "preparador_train.joblib", "base": "codificador_base_train.joblib",
        "district": "codificador_district_train.joblib", "extraccion": "extractor_temporal_train.joblib",
        "filtro": "filtro_features_train.joblib", "escalado": "escalador_train.joblib"}
    for clave, nombre in archivos_objetos.items():
        joblib.dump(objetos_transformacion[clave], artefactos / nombre, compress=3)
    joblib.dump(objetos_transformacion, artefactos / "transformadores_train.joblib", compress=3)

    # Solo exportación de trazabilidad: estos crudos nunca alimentan etapas posteriores.
    train_crudo.to_csv(artefactos / "train_crudo.csv", index=False, date_format="%Y-%m-%d")
    train_crudo.loc[train_crudo["id"].isin(ids_retenidos)].to_csv(artefactos / "train_tukey_crudo.csv", index=False, date_format="%Y-%m-%d")
    train_crudo.loc[~train_crudo["id"].isin(ids_retenidos)].to_csv(artefactos / "train_outliers_tukey.csv", index=False, date_format="%Y-%m-%d")
    train_preparado_antes_tukey.to_csv(artefactos / "train_preparado_antes_tukey.csv", index=False, date_format="%Y-%m-%d")
    train_preparado_antes_tukey.loc[train_actual.index].to_csv(artefactos / "train_preparado_tukey.csv", index=False, date_format="%Y-%m-%d")
    auditoria_train.to_csv(artefactos / "train_auditoria_cambios.csv", index=False)
    pd.DataFrame(trazabilidad_train).to_csv(artefactos / "trazabilidad_train.csv", index=False)
    tabla_estadisticas.to_csv(artefactos / "train_estadisticas_numericas.csv", index_label="variable")
    tabla_decisiones.to_csv(artefactos / "train_decisiones_escalado.csv", index=False)
    pt.guardar_json(artefactos / "tukey_train.json", resumen_tukey)
    resumen_train = pt.exportar_conjunto(train_actual, pt.obtener_X(train_actual), artefactos / "train_modelado.csv")
    nominal = codificador_base.named_transformers_["cat"]
    distrito = codificador_district.named_steps["onehot"]
    contrato_transformacion.update({
        "convencion_nombres": "snake_case_ascii_v1",
        "correspondencia_nombres": filtro_final.named_steps["nombres"].correspondencia_,
        "roles_predictoras": {c: filtro_final.named_steps["nombres"].roles_salida_[c] for c in columnas_modelo},
        "orden_train": [r["etapa"] for r in trazabilidad_train], "particion": contrato_particion,
        "excluidas_modelado": pt.EXCLUIDAS_MODELO, "columnas_X": columnas_modelo,
        "entradas_seleccionadas": pt.ENTRADAS_X,
        "momento_ajuste_imputacion": "Train seleccionado completo, antes de Tukey",
        "medianas_train": dict(zip(pt.NUMERICAS, preparador.numerico_.statistics_.tolist())),
        "medias_train_mom": dict(zip(pt.CAMBIOS_MENSUALES, preparador.mensual_.statistics_.tolist())),
        "modas_binarias_train": dict(zip(pt.BINARIAS, preparador.binario_.statistics_.tolist())),
        "mapas_binarios": pt.MAPAS_BINARIOS,
        "categorias_train": {c: v.tolist() for c,v in zip(pt.CATEGORICAS, nominal.categories_)},
        "categorias_district_train": distrito.categories_[0].tolist(),
        "referencia_district": str(distrito.categories_[0][int(distrito.drop_idx_[0])]),
        "origen_mes_train": extractor_temporal.origen_mes_,
        "tratamiento_objetivo": resumen_tukey, "auditoria_train": resumen_calidad_train,
        "train_modelado": resumen_train,
        "versiones": {p: version(p) for p in ["numpy","pandas","scikit-learn","scipy","joblib"]},
        "modulo_sha256": pt.huella_archivo(SCRIPTS_DIR / "preparacion_train_test.py"),
        "objetos_sha256": {k: pt.huella_archivo(artefactos / v) for k,v in archivos_objetos.items()},
        "flujo_sha256": pt.huella_archivo(artefactos / "transformadores_train.joblib"),
    })
    pt.guardar_json(artefactos / "contrato_transformacion.json", contrato_transformacion)
    objetos_guardados = joblib.load(artefactos / "transformadores_train.joblib")
    huella_antes_test = joblib.hash(objetos_guardados)
    print("Train final:", len(train_actual), "anuncios;", len(columnas_modelo), "predictoras.")
    print(pd.DataFrame(trazabilidad_train)[["etapa","filas_entrada","filas_salida","filas_eliminadas","predictoras"]].to_string(index=False))


Train final: 37984 anuncios; 177 predictoras.
                etapa  filas_entrada  filas_salida  filas_eliminadas  predictoras
            seleccion          39998         39998                 0           20
correccion_imputacion          39998         39998                 0           23
                tukey          39998         37984              2014           23
         codificacion          37984         37984                 0          166
           extraccion          37984         37984                 0          178
         filtro_final          37984         37984                 0          177
      analisis_escala          37984         37984                 0          177
             escalado          37984         37984                 0          177


<a id="seccion-test-final-14"></a>

## 7. Aplicación acumulativa a Test

La función `aplicar_flujo` ejecuta selección → preparación → codificación → extracción →
filtro final → escalado sobre un único DataFrame actualizado. Usa exclusivamente `transform`
con los objetos de Train y omite Tukey. Test conserva sus ID, fechas y precios originales.
El estado de los estimadores, el esquema, las superficies y la finitud deben mantenerse.


In [12]:
with sesion.paso():
    with warnings.catch_warnings(record=True) as avisos_test:
        warnings.simplefilter("always", UserWarning)
        test_actual, resumen_calidad_test, auditoria_test, trazabilidad_test = pt.aplicar_flujo(_test_reservado, objetos_guardados)
    if joblib.hash(objetos_guardados) != huella_antes_test:
        raise ValueError("Test modificó los parámetros de Train.")
    if len(test_actual) != len(_test_reservado):
        raise ValueError("Test perdió anuncios.")
    pt.validar_matriz(pt.obtener_X(test_actual), test_actual, columnas=columnas_modelo)
    auditoria_test.to_csv(artefactos / "test_auditoria_cambios.csv", index=False)
    trazabilidad_test.to_csv(artefactos / "trazabilidad_test.csv", index=False)
    _test_reservado.to_csv(artefactos / "test_crudo.csv", index=False, date_format="%Y-%m-%d")
    resumen_test = pt.exportar_conjunto(test_actual, pt.obtener_X(test_actual), artefactos / "test_modelado.csv")
    pt.guardar_json(artefactos / "aplicacion_test.json", {
        "version_flujo": 3, "flujo_sha256": contrato_transformacion["flujo_sha256"],
        "test_modelado": resumen_test, "auditoria_test": resumen_calidad_test,
        "filtrado_objetivo_aplicado": False, "estimadores_sin_cambios": True,
        "columnas_X_compatibles": pt.obtener_X(test_actual).columns.tolist() == columnas_modelo,
        "avisos_transform": [str(a.message) for a in avisos_test],
    })
    print("Test:", len(test_actual), "anuncios;", len(columnas_modelo), "predictoras compatibles; sin filtrado de precios.")


Test: 10002 anuncios; 177 predictoras compatibles; sin filtrado de precios.


<a id="seccion-exportacion-final-15"></a>

## 8. Exportación de datasets transformados y persistencia de transformadores

Se publican en `Ventas/Propiedades_Usadas/data/` exclusivamente los dos CSV.
Los transformadores y metadatos se registran en MLflow desde temporales del sistema. Cada CSV conserva `price_rub`, las mismas predictoras en el
mismo orden y los auxiliares `id`, `date_posted_dt`, `dataset_split` para trazabilidad;
los auxiliares se excluyen de X mediante `columnas_X` del paquete. Se escribe sin índice.
La reducción de Train por Tukey no redefine el corte 80/20 original. Test conserva todos
sus registros y precios, incluidos los extremos.

Antes de publicar se comprueban identidad, orden, fechas, objetivo, variables descartadas,
valores finitos y esquema. Se recarga el `.joblib` y se reproducen Train retenido y Test
solo con `transform`, verificando igualdad completa y ausencia de cambios en los estimadores.
Los CSV se recargan con `float_precision="round_trip"`. El manifiesto se publica al final
con las huellas SHA-256 de los tres archivos; detecta versiones mezcladas o incompletas.
El módulo `preparacion_train_test.py` y las versiones de librerías registradas son necesarios
para reutilizar el paquete desde un proceso nuevo, sin depender del estado del notebook.


In [13]:
with sesion.paso():
    salida_final = DATOS_PROYECTO
    manifiesto_exportacion = pt.exportar_entrega_final(
        train_actual, test_actual, train_crudo, _test_reservado,
        objetos_guardados, contrato_transformacion, salida_final, carpeta_artefactos=artefactos,
    )
    resumen_exportacion = pd.DataFrame([
        {"conjunto": conjunto, **{k: manifiesto_exportacion[conjunto][k]
         for k in ["archivo", "filas", "columnas", "predictoras", "fecha_min", "fecha_max"]}}
        for conjunto in ["train", "test"]
    ])
    display(resumen_exportacion)
    print("Transformadores recuperados y reproducidos sin fit; manifiesto SHA-256:", salida_final)


conjunto,archivo,filas,columnas,predictoras,fecha_min,fecha_max
train,train_transformado.csv,37984,181,177,2020-01-01,2025-01-16
test,test_transformado.csv,10002,181,177,2025-01-17,2026-04-29


Transformadores recuperados y reproducidos sin fit; manifiesto SHA-256: C:\Users\mglon\OneDrive\Desktop\UNIVERSIDAD\Introduccion al analisis de datos\MOSCOW DATASET\Moscow_EDA\Ventas\Propiedades_Usadas\salidas_transformacion


<a id="seccion-resumen-final-16"></a>

## 9. Resumen final de transformaciones y características del dataset

Auditoría documental generada desde los CSV oficiales recargados, sus huellas y el paquete
ajustado, sin modificar datos ni estimadores. Los grupos One-Hot proceden de las salidas y
categorías reales de cada codificador; su agrupación es solo visual. Las clases numérica,
binaria y One-Hot son disjuntas; las derivadas son un subconjunto de las dos primeras.
La fecha de publicación cuenta como entrada original cuando genera características temporales.
Los motivos de exclusión explícita provienen de la configuración; otras columnas se identifican
como excluidas por la lista blanca, sin atribuirles decisiones metodológicas no implementadas.

El registro completo, con categorías aprendidas, procedencia por columna y orden de X,
se registra como `resumen_final.json` en MLflow desde un archivo temporal. Aquí se muestran tablas compactas.
Este bloque debe cerrar siempre el notebook. El actualizador del índice lo coloca al final y
ajusta su numeración cuando se incorporan nuevas secciones antes de regenerar el índice.


In [14]:
with sesion.paso():
    from IPython.display import Markdown
    from scripts import resumen_transformacion as rt
    rt = importlib.reload(rt)
    estructura_notebook = __import__("json").loads(
        (carpeta_notebook / "2_Transformacion_PropUsadas.ipynb").read_text(encoding="utf-8"))
    if not all(c.get("metadata", {}).get("resumen_final_transformacion")
               for c in estructura_notebook["cells"][-2:]):
        raise ValueError("Actualizar el índice: el resumen debe permanecer al final absoluto.")
    resumen_final = rt.generar_resumen(salida_final, train_crudo, _test_reservado,
        manifiesto=manifiesto_exportacion, objetos=dict(objetos_guardados, configuracion=contrato_transformacion))
    rt.guardar_resumen(resumen_final, artefactos / "resumen_final.json")
    for titulo, tabla in resumen_final["tablas"].items():
        display(Markdown("**" + titulo + "**"))
        with pd.option_context("display.max_rows", None, "display.max_colwidth", 180):
            display(tabla)
    print("Auditoría final conforme: inventario, cantidades y parámetros coinciden con la entrega; sin cambios en datos ni estimadores.")

    from scripts import baseline_ols as b
    cliente = b.conectar(raiz)
    registrar_archivos(cliente, RUN_TRANSFORMACION, artefactos,
        "transformacion/" + manifiesto_exportacion["train"]["sha256"] + "/" + manifiesto_exportacion["modulo_sha256"],
        fuentes=[SCRIPTS_DIR / nombre for nombre in ("__init__.py", "contrato_datasets.py", "preparacion_train_test.py",
            "resumen_transformacion.py", "almacenamiento_mlflow.py")])
    sesion.cerrar()


**Feature Selection: exclusiones y reemplazos**

Variable original,Acción,Motivo / representación
date_posted,Excluida completamente de X,Fuera de la lista blanca de entradas del flujo; no se utiliza en etapas posteriores.
district,Reemplazada por representación transformada,"district_akademichesky, district_alekseevsky, district_altufyevsky, …"
okrug,Excluida completamente de X,Fuera de la lista blanca de entradas del flujo; no se utiliza en etapas posteriores.
lat,Excluida completamente de X,Coordenadas descartadas; se priorizan otras medidas de ubicación y accesibilidad.
lon,Excluida completamente de X,Coordenadas descartadas; se priorizan otras medidas de ubicación y accesibilidad.
total_area,Reemplazada por representación transformada,"total_area, surface_corrected"
living_area,Reemplazada por representación transformada,"living_area, surface_corrected"
kitchen_area,Reemplazada por representación transformada,"kitchen_area, surface_corrected"
rooms,Reemplazada por representación transformada,rooms
floor,Reemplazada por representación transformada,floor


**Feature Transformation: operaciones ejecutadas**

Variables,Método,Aprendizaje,Regla / parámetro,Resultado
total_area,Imputación median,"Train seleccionado completo, antes de Tukey",65.1,Sin faltantes en entradas preparadas; indicadores según imputador ajustado
living_area,Imputación median,"Train seleccionado completo, antes de Tukey",40.6,Sin faltantes en entradas preparadas; indicadores según imputador ajustado
kitchen_area,Imputación median,"Train seleccionado completo, antes de Tukey",11.2,Sin faltantes en entradas preparadas; indicadores según imputador ajustado
rooms,Imputación median,"Train seleccionado completo, antes de Tukey",2.0,Sin faltantes en entradas preparadas; indicadores según imputador ajustado
floor,Imputación median,"Train seleccionado completo, antes de Tukey",6.0,Sin faltantes en entradas preparadas; indicadores según imputador ajustado
total_floors,Imputación median,"Train seleccionado completo, antes de Tukey",12.0,Sin faltantes en entradas preparadas; indicadores según imputador ajustado
building_year,Imputación median,"Train seleccionado completo, antes de Tukey",1988.0,Sin faltantes en entradas preparadas; indicadores según imputador ajustado
ceiling_height,Imputación median,"Train seleccionado completo, antes de Tukey",2.7,Sin faltantes en entradas preparadas; indicadores según imputador ajustado
metro_distance_min,Imputación median,"Train seleccionado completo, antes de Tukey",39.0,Sin faltantes en entradas preparadas; indicadores según imputador ajustado
to_center_km,Imputación median,"Train seleccionado completo, antes de Tukey",13.23,Sin faltantes en entradas preparadas; indicadores según imputador ajustado


**Feature Extraction: derivadas presentes (no One-Hot)**

...

**Dimensiones finales: clases disjuntas y procedencia**

Medida,Cantidad
Registros originales (Train + Test),50000
Train antes de Tukey,39998
Filas eliminadas por Tukey,2014
Train final,37984
Test final (sin filtrado),10002
Columnas predictoras reales,177
Numéricas (excluye binarias y One-Hot),14
"Binarias (incluye indicadores, excluye One-Hot)",5
Columnas One-Hot,158
Derivadas finales (subconjunto de numéricas/binarias),3


**Reglas para futuras observaciones**

Regla,Detalle
Entradas de características,"total_area, living_area, kitchen_area, rooms, floor, total_floors, building_year, ceiling_height, metro_distance_min, to_center_km, metro_to_center_km, mortgage_key_rate_at_listing, secondary_mom_change_pct, building_type, renovation, metro_line, has_balcony, metro_distance_type, seller_type, district"
Auxiliares / respuesta,"Auxiliares: id, date_posted_dt, dataset_split; respuesta separada: price_rub (opcional en inferencia nueva)"
Alias de entrada,"{""mortgage_rate_at_listing"": ""mortgage_key_rate_at_listing""}"
Secuencia,seleccion → correccion_imputacion → tukey → codificacion → extraccion → filtro_final → analisis_escala → escalado
Nuevas observaciones / Test,"aplicar_flujo(nuevos, reglas): solo transform; sin fit ni Tukey; todos los parámetros provienen de Train"
Categorías y orden completo de X,"Categorías completas en resumen_final.json; orden exacto en columnas_X del paquete y del manifiesto, contrastado con CSV (no se imprime el listado One-Hot)"
Persistencia,transformadores_prop_usadas.joblib; district independiente del codificador base; regenerar entrega y resumen ante cambios


Auditoría final conforme: inventario, cantidades y parámetros coinciden con la entrega; sin cambios en datos ni estimadores.
